# 02 — Split spatial CosMx + Xenium avec pseudo-FOV, buffer fin et screening de seeds

Version adaptée : le notebook fait maintenant une **annotation / clustering QC une seule fois sur tout le dataset** avant le screening, puis utilise cette colonne pour calculer la **TVD biologique** pour chaque seed.

Workflow :
1. annotation ou clustering QC global ;
2. stockage dans `obs[CELLTYPE_COL]` ;
3. screening de plusieurs seeds ;
4. calcul de la TVD train vs val/test pour chaque seed ;
5. choix de la seed finale selon tailles, buffer, QC technique et TVD biologique.


Version v4 : la TVD biologique est utilisée comme filtre d'acceptabilité, pas comme critère à minimiser.

In [1]:
from __future__ import annotations

from pathlib import Path
from collections import deque
import warnings

import numpy as np
import pandas as pd
import anndata as ad
from scipy import sparse
from scipy.spatial import cKDTree

import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

KeyboardInterrupt: 

## 0. Paramètres

In [ ]:
from pathlib import Path

# ------------------------------------------------------------
# Chemins Ruche
# ------------------------------------------------------------
ROOT_DIR = Path("/gpfs/workdir/lapielo/projects/clip_baseline")
PROJECT_DIR = ROOT_DIR

DATA_DIR = ROOT_DIR / "data" / "anndata"
OUTPUT_DIR = ROOT_DIR / "outputs" / "split_screening"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "cosmx_breast": {
        "rna_path": DATA_DIR / "cosmx_breast_rna.h5ad",
        "protein_path": DATA_DIR / "cosmx_breast_proteins.h5ad",
        "spatial_scale_to_um": 0.120280945,
        "unit_mode": "native_fov",
        "unit_col": "fov",
    },
    "xenium_renal": {
        "rna_path": DATA_DIR / "xenium_renal_rna.h5ad",
        "protein_path": DATA_DIR / "xenium_renal_proteins_28_marker.h5ad",
        "spatial_scale_to_um": 1.0,
        "unit_mode": "pseudo_fov",
        "unit_col": "pseudo_fov",
        "pseudo_fov_tile_size_um": 800.0,
    },
}

# ------------------------------------------------------------
# Split principal
# ------------------------------------------------------------
SEED = 0

# train = le reste après val/test + buffer.
SPLIT_FRACTIONS = {"val": 0.15, "test": 0.15}
CELL_BUFFER_WIDTH_UM = 100.0
CONNECTIVITY_RADIUS_FACTOR = 1.25
ENFORCE_CONNECTED_COMPONENTS = True
OVERWRITE_SPATIAL_WITH_UM = True

# ------------------------------------------------------------
# Screening de seeds
# ------------------------------------------------------------
RUN_SEED_SCREENING = True
RUN_FINAL_SPLIT_AFTER_SCREENING = True

# Si FINAL_SEED = None : le notebook prend automatiquement la meilleure seed selon le QC du split.
# Si tu veux figer une seed manuellement, mets par exemple FINAL_SEED = 42.
FINAL_SEED = None
SEEDS_TO_TEST = [0, 1, 2, 3, 4, 5, 10, 20, 42, 100]

RANDOMIZE_TARGETS_BY_SEED = True
TARGET_MARGIN_NORM = 0.12
MIN_TARGET_DISTANCE_NORM = 0.45

# Critères pratiques pour le score de QC du split.
MAX_ACCEPTABLE_BUFFER_PCT = 20.0
MIN_ACCEPTABLE_VAL_TEST_PCT = 7.0
MAX_ACCEPTABLE_TECH_SHIFT_PCT = 30.0

# ------------------------------------------------------------
# Annotation / clustering QC avant screening de seeds
# ------------------------------------------------------------
RUN_GLOBAL_CPU_ANNOTATION_FOR_SEED_SCREENING = True

EXISTING_CELLTYPE_CANDIDATES = [
    "cell_type", "celltype", "annotation", "cell_type_scconcept",
    "qc_celltype", "qc_celltype_cpu", "leiden_cpu",
]

CELLTYPE_COL = "qc_celltype_cpu"
QC_LEIDEN_COL = "qc_leiden_cpu"

LEIDEN_RESOLUTION = 1.0
CPU_ANNOTATION_N_HVG = 1_000
CPU_ANNOTATION_N_PCS = 30
CPU_ANNOTATION_N_NEIGHBORS = 15
FORCE_RECOMPUTE_CELLTYPE_QC = False

# TVD biologique : utilisée comme filtre QC, pas comme objectif à minimiser.
CELLTYPE_TVD_WARNING = 0.15
MAX_ACCEPTABLE_CELLTYPE_TVD = 0.20
USE_CELLTYPE_TVD_AS_FILTER_ONLY = True
SAVE_GLOBAL_CELLTYPE_QC = True

# ------------------------------------------------------------
# Affichage
# ------------------------------------------------------------
DISPLAY_MODE = "compact"
DISPLAY_QC_TABLES = False
DISPLAY_CELLTYPE_TABLES = False

# Sur Ruche en batch, je conseille False pour éviter un notebook exécuté trop lourd.
# Les figures seront quand même sauvegardées si ton code appelle plt.savefig().
SHOW_PLOTS = False

MAKE_TECHNICAL_BOXPLOTS = False
SAVE_H5AD = True

PLOT_MAX_CELLS = 200_000
PLOT_POINT_SIZE = 1.0
HIST_BINS = 60

SPLIT_ORDER = ["train", "val", "test", "buffer"]

SPLIT_COLORS = {
    "train": "tab:blue",
    "val": "tab:orange",
    "test": "tab:green",
    "buffer": "lightgray",
}

TECHNICAL_OBS_KEYWORDS = [
    "total_counts",
    "n_genes",
    "n_genes_by_counts",
    "pct_counts",
    "area",
    "cell_area",
    "nucleus_area",
    "transcript",
    "count",
    "umi",
]

# ------------------------------------------------------------
# Log de progression pour exécution Slurm / nbconvert
# ------------------------------------------------------------
# nbconvert ne remplit pas toujours le .out en direct.
# Ce fichier permet de suivre le notebook avec :
# tail -f /gpfs/workdir/lapielo/projects/clip_baseline/outputs/split_screening/progress.log

PROGRESS_LOG = OUTPUT_DIR / "progress.log"

def log_progress(msg):
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {msg}"
    print(line, flush=True)
    with open(PROGRESS_LOG, "a", encoding="utf-8") as f:
        f.write(line + "\n")
        f.flush()

# Réinitialise le log à chaque exécution complète du notebook
with open(PROGRESS_LOG, "w", encoding="utf-8") as f:
    f.write("==== Nouveau run split screening ====\n")

log_progress("Paramètres chargés.")
log_progress(f"ROOT_DIR = {ROOT_DIR}")
log_progress(f"DATA_DIR = {DATA_DIR}")
log_progress(f"OUTPUT_DIR = {OUTPUT_DIR}")
log_progress(f"CELL_BUFFER_WIDTH_UM = {CELL_BUFFER_WIDTH_UM}")
log_progress(f"RUN_SEED_SCREENING = {RUN_SEED_SCREENING}")
log_progress(f"DISPLAY_MODE = {DISPLAY_MODE}")


Paramètres chargés.
ROOT_DIR = /Users/louiselapie/Documents/creation_dataset/anndata_v2
OUTPUT_DIR = outputs_split_cosmx_xenium_pseudofov_cell_boundary_buffer_qc
CELL_BUFFER_WIDTH_UM = 100.0
RUN_SEED_SCREENING = True
DISPLAY_MODE = compact


## 1. Fonctions utilitaires

In [ ]:

def as_dense_array(x):
    return x.toarray() if sparse.issparse(x) else np.asarray(x)


def row_sums_and_detected(adata: ad.AnnData) -> tuple[np.ndarray, np.ndarray]:
    X = adata.X
    if sparse.issparse(X):
        totals = np.asarray(X.sum(axis=1)).ravel()
        detected = np.asarray((X > 0).sum(axis=1)).ravel()
    else:
        X = np.asarray(X)
        totals = X.sum(axis=1)
        detected = (X > 0).sum(axis=1)
    return totals.astype(float), detected.astype(float)


def harmonize_pair(rna: ad.AnnData, prot: ad.AnnData) -> tuple[ad.AnnData, ad.AnnData]:
    """Aligne RNA et protéine sur les obs_names communs, dans le même ordre."""
    if (rna.obs_names == prot.obs_names).all():
        return rna, prot

    common = rna.obs_names.intersection(prot.obs_names)
    log_progress(f"[INFO] obs_names non identiques. Intersection utilisée : {len(common):,} cellules")
    if len(common) == 0:
        raise ValueError("Aucune cellule commune entre RNA et protéine.")

    rna2 = rna[common].copy()
    prot2 = prot[common].copy()
    assert (rna2.obs_names == prot2.obs_names).all()
    return rna2, prot2


def get_spatial_input(adata: ad.AnnData) -> tuple[np.ndarray, str]:
    """Récupère des coordonnées spatiales depuis obsm['spatial'] ou des colonnes obs fréquentes."""
    if "spatial" in adata.obsm:
        coords = np.asarray(adata.obsm["spatial"])
        if coords.ndim == 2 and coords.shape[1] >= 2:
            return coords[:, :2].astype(float), "obsm['spatial']"

    candidates = [
        ("x", "y"), ("X", "Y"),
        ("x_centroid", "y_centroid"), ("centroid_x", "centroid_y"),
        ("CenterX_global_px", "CenterY_global_px"),
        ("CenterX_local_px", "CenterY_local_px"),
        ("x_location", "y_location"),
    ]
    for x_col, y_col in candidates:
        if x_col in adata.obs.columns and y_col in adata.obs.columns:
            coords = adata.obs[[x_col, y_col]].to_numpy(dtype=float)
            return coords[:, :2], f"obs[['{x_col}', '{y_col}']]"

    raise ValueError("Impossible de trouver des coordonnées spatiales.")


def prepare_spatial_um(rna: ad.AnnData, prot: ad.AnnData, cfg: dict, dataset: str) -> tuple[ad.AnnData, ad.AnnData]:
    """Crée obsm['spatial_um'] pour RNA et protéine, et optionnellement remplace obsm['spatial']."""
    coords_raw, source = get_spatial_input(rna)
    scale = float(cfg.get("spatial_scale_to_um", 1.0))
    coords_um = coords_raw * scale

    for a in [rna, prot]:
        if "spatial_raw_input" not in a.obsm:
            # On suppose que prot a les mêmes obs_names, donc même ordre que rna.
            a.obsm["spatial_raw_input"] = coords_raw.copy()
        a.obsm["spatial_um"] = coords_um.copy()
        if OVERWRITE_SPATIAL_WITH_UM:
            a.obsm["spatial"] = coords_um.copy()
        a.obs["x_um"] = coords_um[:, 0]
        a.obs["y_um"] = coords_um[:, 1]

    log_progress(f"[{dataset}] Coordonnées : {source}")
    log_progress(f"[{dataset}] Facteur conversion vers µm : {scale}")
    log_progress(
        f"[{dataset}] bbox µm : "
        f"x=[{coords_um[:,0].min():.1f}, {coords_um[:,0].max():.1f}], "
        f"y=[{coords_um[:,1].min():.1f}, {coords_um[:,1].max():.1f}]"
    )
    return rna, prot

## 2. Unités spatiales : FOV CosMx et pseudo-FOV Xenium

In [ ]:
def add_pseudo_fov_from_grid(
    rna: ad.AnnData,
    prot: ad.AnnData,
    unit_col: str,
    tile_size_um: float,
) -> tuple[ad.AnnData, ad.AnnData]:
    """Crée des pseudo-FOV par tuilage régulier des coordonnées en µm."""
    coords = np.asarray(rna.obsm["spatial_um"])
    x0, y0 = coords[:, 0].min(), coords[:, 1].min()
    ix = np.floor((coords[:, 0] - x0) / tile_size_um).astype(int)
    iy = np.floor((coords[:, 1] - y0) / tile_size_um).astype(int)

    pseudo = pd.Series([f"tile_{i:03d}_{j:03d}" for i, j in zip(ix, iy)], index=rna.obs_names, name=unit_col)

    for a in [rna, prot]:
        a.obs[unit_col] = pseudo.loc[a.obs_names].astype(str).values
        a.obs[f"{unit_col}_ix"] = ix
        a.obs[f"{unit_col}_iy"] = iy

    log_progress(f"Pseudo-FOV créés : {pseudo.nunique()} tuiles de {tile_size_um:.0f} µm")
    return rna, prot


def build_unit_positions(adata: ad.AnnData, unit_col: str) -> pd.DataFrame:
    """Construit la table des unités spatiales avec position médiane et nombre de cellules."""
    if unit_col not in adata.obs.columns:
        raise ValueError(f"Colonne unité absente : {unit_col}")

    tmp = pd.DataFrame({
        "unit_id": adata.obs[unit_col].astype(str).to_numpy(),
        "x_um": adata.obsm["spatial_um"][:, 0],
        "y_um": adata.obsm["spatial_um"][:, 1],
    }, index=adata.obs_names)

    unit_pos = (
        tmp.groupby("unit_id")
        .agg(
            x_um=("x_um", "median"),
            y_um=("y_um", "median"),
            n_cells=("x_um", "size"),
        )
        .reset_index()
    )

    return unit_pos


def add_normalized_coords(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for axis in ["x_um", "y_um"]:
        mn = df[axis].min()
        mx = df[axis].max()
        denom = mx - mn
        if denom == 0:
            df[axis.replace("_um", "_norm")] = 0.5
        else:
            df[axis.replace("_um", "_norm")] = (df[axis] - mn) / denom
    return df


def decide_n_units(cfg: dict, n_units_total: int, key: str) -> int:
    """Récupère n_val_units/n_test_units ou calcule depuis une fraction."""
    fixed = cfg.get(f"n_{key}_units", None)
    if fixed is not None:
        return int(fixed)

    frac = float(cfg.get(f"{key}_fraction_units", 0.10))
    n = int(round(frac * n_units_total))
    n = max(int(cfg.get(f"min_{key}_units", 1)), n)
    n = min(int(cfg.get(f"max_{key}_units", n)), n)
    n = min(n, n_units_total)
    return int(n)

## 3. Fonctions de split spatial

In [ ]:
def select_compact_block(
    unit_pos: pd.DataFrame,
    target_norm: tuple[float, float],
    n_units: int,
    forbidden_units: set[str] | None = None,
) -> set[str]:
    """Sélectionne les n unités les plus proches d'une cible normalisée."""
    forbidden_units = set() if forbidden_units is None else set(map(str, forbidden_units))
    df = unit_pos.copy()
    allowed = ~df["unit_id"].astype(str).isin(forbidden_units)
    df = df.loc[allowed].copy()
    if df.empty:
        raise ValueError("Aucune unité disponible pour sélectionner un bloc.")

    tx, ty = target_norm
    df["dist_target"] = np.sqrt((df["x_norm"] - tx) ** 2 + (df["y_norm"] - ty) ** 2)
    chosen = df.sort_values("dist_target").head(n_units)["unit_id"].astype(str).tolist()
    return set(chosen)


def select_compact_block_by_cells(
    unit_pos: pd.DataFrame,
    target_norm: tuple[float, float],
    target_frac_cells: float,
    total_cells: int,
    forbidden_units: set[str] | None = None,
) -> set[str]:
    """Sélectionne un bloc compact d'unités autour d'une cible normalisée.

    Les unités les plus proches de la cible sont accumulées jusqu'à atteindre
    une fraction cible du nombre total de cellules.
    """
    forbidden_units = set() if forbidden_units is None else set(map(str, forbidden_units))
    df = unit_pos.copy()
    df = df.loc[~df["unit_id"].astype(str).isin(forbidden_units)].copy()
    if df.empty:
        raise ValueError("Aucune unité disponible pour sélectionner un bloc.")

    tx, ty = target_norm
    df["dist_target"] = np.sqrt((df["x_norm"] - tx) ** 2 + (df["y_norm"] - ty) ** 2)
    df = df.sort_values("dist_target")

    target_cells = target_frac_cells * total_cells
    chosen, cum = [], 0
    for _, row in df.iterrows():
        chosen.append(str(row["unit_id"]))
        cum += int(row["n_cells"])
        if cum >= target_cells:
            break
    return set(chosen)


def build_unit_adjacency(unit_pos: pd.DataFrame, radius_factor: float = 1.25) -> tuple[dict[str, set[str]], float]:
    """Construit un graphe d'adjacence entre unités à partir des positions médianes."""
    ids = unit_pos["unit_id"].astype(str).tolist()
    coords = unit_pos[["x_um", "y_um"]].to_numpy(float)
    tree = cKDTree(coords)

    if len(coords) <= 1:
        return {ids[0]: set()} if ids else {}, 0.0

    nn_dist, _ = tree.query(coords, k=2)
    median_nn = float(np.median(nn_dist[:, 1]))
    radius = median_nn * radius_factor

    neigh = tree.query_ball_point(coords, r=radius)
    adjacency = {u: set() for u in ids}
    for i, js in enumerate(neigh):
        for j in js:
            if i == j:
                continue
            adjacency[ids[i]].add(ids[j])
            adjacency[ids[j]].add(ids[i])
    return adjacency, radius


def connected_components(units: set[str], adjacency: dict[str, set[str]]) -> list[set[str]]:
    units = set(map(str, units))
    seen = set()
    comps = []
    for start in sorted(units):
        if start in seen:
            continue
        q = deque([start])
        seen.add(start)
        comp = {start}
        while q:
            u = q.popleft()
            for v in adjacency.get(u, set()):
                if v in units and v not in seen:
                    seen.add(v)
                    comp.add(v)
                    q.append(v)
        comps.append(comp)
    return comps


def component_size_cells(unit_pos: pd.DataFrame, comp: set[str]) -> int:
    return int(unit_pos.loc[unit_pos["unit_id"].astype(str).isin(comp), "n_cells"].sum())


def enforce_largest_connected_component_per_split(
    unit_pos: pd.DataFrame,
    unit_split: pd.Series,
    adjacency: dict[str, set[str]],
    splits_to_clean: tuple[str, ...] = ("val", "test"),
) -> tuple[pd.Series, pd.DataFrame]:
    """Garde la plus grande composante connexe pour val/test et renvoie les îlots vers train."""
    unit_split = unit_split.copy().astype(str)
    report_rows = []
    dropped_units = set()

    for split in splits_to_clean:
        units = set(unit_split.index[unit_split == split].astype(str))
        comps = connected_components(units, adjacency)
        comps = sorted(comps, key=lambda c: (component_size_cells(unit_pos, c), len(c)), reverse=True)

        for i, comp in enumerate(comps):
            report_rows.append({
                "split": split,
                "component_rank": i,
                "n_units": len(comp),
                "n_cells": component_size_cells(unit_pos, comp),
                "action": "keep" if i == 0 else "reassign_to_train",
            })

        for comp in comps[1:]:
            dropped_units.update(comp)

    if dropped_units:
        # Les îlots déconnectés de val/test sont renvoyés vers train.
        # On ne force pas le train à être connexe : train = reste de la slide.
        for unit_id in dropped_units:
            unit_split.loc[str(unit_id)] = "train"

    return unit_split, pd.DataFrame(report_rows)


def sample_split_targets_from_seed(
    seed: int,
    margin: float = TARGET_MARGIN_NORM,
    min_distance: float = MIN_TARGET_DISTANCE_NORM,
    max_tries: int = 1000,
) -> tuple[tuple[float, float], tuple[float, float]]:
    """Génère deux centres normalisés pour val/test à partir d'une seed.

    Dans l'ancien notebook, les centres val/test étaient fixes, donc changer SEED
    ne changeait pratiquement pas le split. Ici, la seed sert à tester plusieurs
    positions compactes de val/test, uniquement pour le QC du split.
    """
    rng = np.random.default_rng(seed)
    low, high = margin, 1.0 - margin

    for _ in range(max_tries):
        val_target = rng.uniform(low, high, size=2)
        test_target = rng.uniform(low, high, size=2)
        if np.linalg.norm(val_target - test_target) >= min_distance:
            return tuple(val_target), tuple(test_target)

    raise ValueError(
        "Impossible de générer deux centres val/test assez éloignés. "
        "Diminue MIN_TARGET_DISTANCE_NORM ou TARGET_MARGIN_NORM."
    )


def make_unit_split(
    unit_pos: pd.DataFrame,
    cfg: dict,
    seed: int | None = None,
    randomize_targets: bool | None = None,
) -> tuple[pd.Series, pd.DataFrame, dict]:
    """Crée un split de base au niveau unité spatiale : train / val / test."""
    unit_pos = add_normalized_coords(unit_pos)
    unit_pos["unit_id"] = unit_pos["unit_id"].astype(str)

    n_total = len(unit_pos)
    total_cells = int(unit_pos["n_cells"].sum())
    val_frac = float(cfg.get("val_frac_cells", SPLIT_FRACTIONS["val"]))
    test_frac = float(cfg.get("test_frac_cells", SPLIT_FRACTIONS["test"]))

    if randomize_targets is None:
        randomize_targets = RANDOMIZE_TARGETS_BY_SEED

    if randomize_targets:
        if seed is None:
            seed = SEED
        val_target_norm, test_target_norm = sample_split_targets_from_seed(seed)
    else:
        val_target_norm = tuple(cfg["val_target_norm"])
        test_target_norm = tuple(cfg["test_target_norm"])

    val_units = select_compact_block_by_cells(
        unit_pos,
        target_norm=val_target_norm,
        target_frac_cells=val_frac,
        total_cells=total_cells,
        forbidden_units=set(),
    )

    test_units = select_compact_block_by_cells(
        unit_pos,
        target_norm=test_target_norm,
        target_frac_cells=test_frac,
        total_cells=total_cells,
        forbidden_units=val_units,
    )

    unit_split = pd.Series("train", index=unit_pos["unit_id"].astype(str), name="unit_base_split")
    unit_split.loc[list(val_units)] = "val"
    unit_split.loc[list(test_units)] = "test"

    adjacency, radius_um = build_unit_adjacency(unit_pos, radius_factor=CONNECTIVITY_RADIUS_FACTOR)

    if ENFORCE_CONNECTED_COMPONENTS:
        unit_split, cc_report = enforce_largest_connected_component_per_split(
            unit_pos=unit_pos,
            unit_split=unit_split,
            adjacency=adjacency,
            splits_to_clean=("val", "test"),
        )
    else:
        cc_report = pd.DataFrame()

    info = {
        "seed": seed,
        "randomize_targets": bool(randomize_targets),
        "val_target_norm": tuple(float(x) for x in val_target_norm),
        "test_target_norm": tuple(float(x) for x in test_target_norm),
        "n_units_total": n_total,
        "n_val_units_initial": len(val_units),
        "n_test_units_initial": len(test_units),
        "val_frac_cells_target": val_frac,
        "test_frac_cells_target": test_frac,
        "total_cells": total_cells,
        "connectivity_radius_um": radius_um,
    }
    return unit_split, cc_report, info


def apply_base_split_to_cells(rna: ad.AnnData, prot: ad.AnnData, unit_col: str, unit_split: pd.Series) -> tuple[ad.AnnData, ad.AnnData]:
    unit_split = unit_split.astype(str)
    for a in [rna, prot]:
        units = a.obs[unit_col].astype(str)
        a.obs["unit_base_split"] = units.map(unit_split).astype(str).values
        if a.obs["unit_base_split"].isna().any():
            raise ValueError("Certaines cellules n'ont pas reçu de unit_base_split.")
    return rna, prot


def add_cell_level_boundary_buffer(
    rna: ad.AnnData,
    prot: ad.AnnData,
    width_um: float,
    base_split_col: str = "unit_base_split",
    output_col: str = "split",
) -> tuple[ad.AnnData, ad.AnnData, dict]:
    """Crée un buffer fin au niveau cellule autour des frontières entre splits de base."""
    coords = np.asarray(rna.obsm["spatial_um"])
    base = rna.obs[base_split_col].astype(str).to_numpy()
    final = base.copy()
    buffer_mask = np.zeros(rna.n_obs, dtype=bool)

    split_pairs = [("train", "val"), ("train", "test"), ("val", "test")]
    pair_counts = {}

    for a_split, b_split in split_pairs:
        idx_a = np.where(base == a_split)[0]
        idx_b = np.where(base == b_split)[0]
        if len(idx_a) == 0 or len(idx_b) == 0:
            pair_counts[f"{a_split}_vs_{b_split}"] = 0
            continue

        tree_b = cKDTree(coords[idx_b])
        tree_a = cKDTree(coords[idx_a])

        dist_a, _ = tree_b.query(coords[idx_a], k=1, distance_upper_bound=width_um)
        dist_b, _ = tree_a.query(coords[idx_b], k=1, distance_upper_bound=width_um)

        near_a = idx_a[np.isfinite(dist_a) & (dist_a <= width_um)]
        near_b = idx_b[np.isfinite(dist_b) & (dist_b <= width_um)]
        pair_mask_indices = np.concatenate([near_a, near_b])
        buffer_mask[pair_mask_indices] = True
        pair_counts[f"{a_split}_vs_{b_split}"] = int(len(np.unique(pair_mask_indices)))

    final[buffer_mask] = "buffer"

    for a in [rna, prot]:
        a.obs[output_col] = pd.Categorical(final, categories=SPLIT_ORDER, ordered=True)

    info = {
        "buffer_width_um": width_um,
        "n_buffer_cells": int(buffer_mask.sum()),
        "pct_buffer_cells": float(100 * buffer_mask.mean()),
        **pair_counts,
    }
    return rna, prot, info


## 4. Fonctions QC et visualisation

In [ ]:
def split_counts(adata: ad.AnnData, split_col: str = "split") -> pd.DataFrame:
    vc = adata.obs[split_col].astype(str).value_counts().reindex(SPLIT_ORDER).fillna(0).astype(int)
    out = pd.DataFrame({"n_cells": vc, "pct_cells": 100 * vc / adata.n_obs})
    return out.round({"pct_cells": 2})


def unit_counts(adata: ad.AnnData, unit_col: str, split_col: str = "unit_base_split") -> pd.DataFrame:
    tmp = adata.obs[[unit_col, split_col]].drop_duplicates().copy()
    vc = tmp[split_col].astype(str).value_counts().reindex(["train", "val", "test"]).fillna(0).astype(int)
    out = pd.DataFrame({"n_units": vc, "pct_units": 100 * vc / len(tmp)})
    return out.round({"pct_units": 2})


def get_obs_technical_metrics(adata: ad.AnnData, modality: str) -> pd.DataFrame:
    totals, detected = row_sums_and_detected(adata)
    df = pd.DataFrame(index=adata.obs_names)
    df[f"{modality}_total_X"] = totals
    df[f"{modality}_detected_X"] = detected

    # Ajoute les colonnes obs numériques qui semblent techniques.
    for col in adata.obs.columns:
        low = col.lower()
        if any(k in low for k in TECHNICAL_OBS_KEYWORDS):
            vals = pd.to_numeric(adata.obs[col], errors="coerce")
            if vals.notna().sum() > 0 and col not in df.columns:
                df[col] = vals.values
    return df


def summarize_metrics_by_split(metrics: pd.DataFrame, split: pd.Series) -> pd.DataFrame:
    tmp = metrics.copy()
    tmp["split"] = split.astype(str).values
    rows = []
    for split_name in SPLIT_ORDER:
        sub = tmp[tmp["split"] == split_name]
        if sub.empty:
            continue
        row = {"split": split_name, "n_cells": len(sub)}
        for col in metrics.columns:
            x = pd.to_numeric(sub[col], errors="coerce")
            if x.notna().sum() == 0:
                continue
            row[f"{col}__median"] = float(np.nanmedian(x))
            row[f"{col}__mean"] = float(np.nanmean(x))
        rows.append(row)
    return pd.DataFrame(rows)


def median_shift_report(summary: pd.DataFrame, reference_split: str = "train") -> pd.DataFrame:
    if summary.empty or reference_split not in summary["split"].values:
        return pd.DataFrame()
    ref = summary.set_index("split").loc[reference_split]
    rows = []
    median_cols = [c for c in summary.columns if c.endswith("__median")]
    for _, row in summary.iterrows():
        split = row["split"]
        if split == reference_split or split == "buffer":
            continue
        for col in median_cols:
            ref_val = ref[col]
            val = row[col]
            if not np.isfinite(ref_val) or not np.isfinite(val) or ref_val == 0:
                continue
            rel = 100 * (val - ref_val) / abs(ref_val)
            rows.append({
                "split": split,
                "metric": col.replace("__median", ""),
                "train_median": ref_val,
                "split_median": val,
                "relative_shift_pct_vs_train": rel,
                "flag_abs_shift_gt_30pct": abs(rel) > MAX_ACCEPTABLE_TECH_SHIFT_PCT,
            })
    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(["flag_abs_shift_gt_30pct", "split", "metric"], ascending=[False, True, True])
    return out


def count_shift_flags(shift: pd.DataFrame) -> int:
    if shift is None or shift.empty or "flag_abs_shift_gt_30pct" not in shift.columns:
        return 0
    return int(shift["flag_abs_shift_gt_30pct"].sum())


def max_abs_shift_pct(shift: pd.DataFrame) -> float:
    if shift is None or shift.empty or "relative_shift_pct_vs_train" not in shift.columns:
        return np.nan
    return float(shift["relative_shift_pct_vs_train"].abs().max())


def plot_spatial_split(
    adata: ad.AnnData,
    title: str,
    path: Path,
    split_col: str = "split",
    max_cells: int | None = PLOT_MAX_CELLS,
    show: bool = True,
):
    obs = adata.obs.copy()
    coords = np.asarray(adata.obsm["spatial_um"])
    obs["x_um"] = coords[:, 0]
    obs["y_um"] = coords[:, 1]

    if max_cells is not None and len(obs) > max_cells:
        obs = obs.sample(max_cells, random_state=SEED)

    fig, ax = plt.subplots(figsize=(7, 7))
    for split in SPLIT_ORDER:
        sub = obs[obs[split_col].astype(str) == split]
        if sub.empty:
            continue
        ax.scatter(
            sub["x_um"], sub["y_um"],
            s=PLOT_POINT_SIZE,
            alpha=0.65 if split != "buffer" else 0.4,
            label=f"{split} ({len(sub):,})",
            c=SPLIT_COLORS.get(split, None),
            linewidths=0,
        )
    ax.set_title(title)
    ax.set_xlabel("x (µm)")
    ax.set_ylabel("y (µm)")
    ax.set_aspect("equal", adjustable="box")
    ax.invert_yaxis()
    ax.legend(markerscale=5, frameon=False, loc="best")
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig


def plot_unit_split(unit_pos: pd.DataFrame, unit_split: pd.Series, title: str, path: Path, show: bool = True):
    df = unit_pos.copy()
    df["unit_base_split"] = df["unit_id"].astype(str).map(unit_split).astype(str)
    fig, ax = plt.subplots(figsize=(7, 7))
    for split in ["train", "val", "test"]:
        sub = df[df["unit_base_split"] == split]
        if sub.empty:
            continue
        ax.scatter(
            sub["x_um"], sub["y_um"],
            s=np.clip(sub["n_cells"] / sub["n_cells"].median() * 30, 10, 100),
            alpha=0.85,
            label=f"{split} ({len(sub)} unités)",
            c=SPLIT_COLORS.get(split, None),
            edgecolors="black",
            linewidths=0.3,
        )
    ax.set_title(title)
    ax.set_xlabel("x (µm)")
    ax.set_ylabel("y (µm)")
    ax.set_aspect("equal", adjustable="box")
    ax.invert_yaxis()
    ax.legend(frameon=False, loc="best")
    fig.tight_layout()
    fig.savefig(path, dpi=200)
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig


def plot_metric_boxplots(metrics: pd.DataFrame, split: pd.Series, title: str, path: Path, max_metrics: int = 6, show: bool = True):
    cols = list(metrics.columns[:max_metrics])
    if len(cols) == 0:
        return None

    tmp = metrics[cols].copy()
    tmp["split"] = split.astype(str).values

    fig, axes = plt.subplots(len(cols), 1, figsize=(8, 2.6 * len(cols)))
    if len(cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, cols):
        data = [tmp.loc[tmp["split"] == s, col].dropna().values for s in SPLIT_ORDER if (tmp["split"] == s).any()]
        labels = [s for s in SPLIT_ORDER if (tmp["split"] == s).any()]
        ax.boxplot(data, labels=labels, showfliers=False)
        ax.set_title(col)
        ax.set_ylabel("value")
    fig.suptitle(title, y=1.01)
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)
    return fig


## 5. Pipeline par dataset

In [ ]:


def make_h5ad_safe(value):
    """
    Convertit récursivement les objets qui posent problème dans .uns
    lors de l'écriture en .h5ad.

    AnnData/HDF5 n'aime notamment pas les tuples dans .uns.
    On convertit donc tuples, Path et scalaires numpy en objets simples.
    """
    from pathlib import Path
    import numpy as np

    if isinstance(value, dict):
        return {str(k): make_h5ad_safe(v) for k, v in value.items()}
    if isinstance(value, tuple):
        return [make_h5ad_safe(v) for v in value]
    if isinstance(value, list):
        return [make_h5ad_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if value is None:
        return "None"
    return value

def process_one_dataset(
    dataset: str,
    cfg: dict,
    seed: int | None = None,
    display_mode: str = DISPLAY_MODE,
    make_plots: bool = SHOW_PLOTS,
    save_h5ad: bool = SAVE_H5AD,
):
    """Pipeline complet pour un dataset avec affichage compact par défaut."""
    if seed is None:
        seed = SEED

    verbose = display_mode == "full"

    dataset_out = OUTPUT_DIR / dataset
    fig_dir = dataset_out / "figures"
    table_dir = dataset_out / "tables"
    h5ad_dir = dataset_out / "h5ad"
    for d in [fig_dir, table_dir, h5ad_dir]:
        d.mkdir(parents=True, exist_ok=True)

    display(Markdown(f"## Dataset final : `{dataset}` — seed `{seed}`"))
    if verbose:
        log_progress(f"RNA path     : {cfg['rna_path']}")
        log_progress(f"Protein path : {cfg['protein_path']}")

    rna = ad.read_h5ad(cfg["rna_path"])
    prot = ad.read_h5ad(cfg["protein_path"])
    log_progress(f"RNA     : {rna.n_obs:,} cellules × {rna.n_vars:,} variables")
    log_progress(f"Protein : {prot.n_obs:,} cellules × {prot.n_vars:,} variables")

    rna, prot = harmonize_pair(rna, prot)
    rna, prot = prepare_spatial_um(rna, prot, cfg, dataset)

    unit_col = cfg["unit_col"]
    if cfg["unit_mode"] == "pseudo_fov":
        rna, prot = add_pseudo_fov_from_grid(
            rna,
            prot,
            unit_col=unit_col,
            tile_size_um=float(cfg["pseudo_fov_tile_size_um"]),
        )
    elif cfg["unit_mode"] == "native_fov":
        if unit_col not in rna.obs.columns:
            raise ValueError(f"{dataset}: colonne FOV native absente : {unit_col}")
        for a in [rna, prot]:
            a.obs[unit_col] = a.obs[unit_col].astype(str)
        log_progress(f"FOV natifs trouvés : {rna.obs[unit_col].nunique()} unités")
    else:
        raise ValueError(f"unit_mode inconnu : {cfg['unit_mode']}")

    unit_pos = build_unit_positions(rna, unit_col=unit_col)
    unit_pos.to_csv(table_dir / f"{dataset}_unit_positions.csv", index=False)
    if verbose:
        display(unit_pos.describe(include="all"))

    unit_split, cc_report, split_info = make_unit_split(unit_pos, cfg, seed=seed)
    cc_report.to_csv(table_dir / f"{dataset}_connected_components_report.csv", index=False)

    log_progress("Cibles normalisées val/test :")
    log_progress(f"  val  : {tuple(round(x, 3) for x in split_info['val_target_norm'])}")
    log_progress(f"  test : {tuple(round(x, 3) for x in split_info['test_target_norm'])}")

    if verbose:
        log_progress("Informations split unités :")
        for k, v in split_info.items():
            log_progress(f"  {k}: {v}")
        display(Markdown("### Split par unités spatiales, avant buffer cellule"))
        display(pd.DataFrame({"unit_base_split": unit_split}).value_counts().rename("n_units").reset_index())

    plot_unit_split(
        unit_pos,
        unit_split,
        title=f"{dataset} — split des unités spatiales — seed {seed}",
        path=fig_dir / f"{dataset}_01_unit_split_seed{seed}.png",
        show=make_plots,
    )

    rna, prot = apply_base_split_to_cells(rna, prot, unit_col=unit_col, unit_split=unit_split)
    rna, prot, buffer_info = add_cell_level_boundary_buffer(
        rna,
        prot,
        width_um=CELL_BUFFER_WIDTH_UM,
        base_split_col="unit_base_split",
        output_col="split",
    )

    split_summary = split_counts(rna, split_col="split")
    split_summary.to_csv(table_dir / f"{dataset}_split_cell_counts.csv")

    display(Markdown("### Résumé final par cellule"))
    display(split_summary)

    if verbose:
        log_progress("Informations buffer cellule :")
        for k, v in buffer_info.items():
            log_progress(f"  {k}: {v}")
        display(Markdown("### Pourcentages par unité spatiale"))
        unit_summary = unit_counts(rna, unit_col=unit_col, split_col="unit_base_split")
        display(unit_summary)
    else:
        unit_summary = unit_counts(rna, unit_col=unit_col, split_col="unit_base_split")

    unit_summary.to_csv(table_dir / f"{dataset}_unit_base_split_counts.csv")

    plot_spatial_split(
        rna,
        title=f"{dataset} — split final avec buffer cellule — seed {seed}",
        path=fig_dir / f"{dataset}_02_cell_split_final_seed{seed}.png",
        split_col="split",
        show=make_plots,
    )

    # QC distributions techniques RNA
    rna_metrics = get_obs_technical_metrics(rna, modality="rna")
    rna_metric_summary = summarize_metrics_by_split(rna_metrics, rna.obs["split"])
    rna_metric_summary.to_csv(table_dir / f"{dataset}_rna_technical_metrics_by_split.csv", index=False)

    rna_shift = median_shift_report(rna_metric_summary, reference_split="train")
    rna_shift.to_csv(table_dir / f"{dataset}_rna_median_shift_vs_train.csv", index=False)

    # QC distributions techniques protéine
    prot_metrics = get_obs_technical_metrics(prot, modality="protein")
    prot_metric_summary = summarize_metrics_by_split(prot_metrics, prot.obs["split"])
    prot_metric_summary.to_csv(table_dir / f"{dataset}_protein_technical_metrics_by_split.csv", index=False)

    prot_shift = median_shift_report(prot_metric_summary, reference_split="train")
    prot_shift.to_csv(table_dir / f"{dataset}_protein_median_shift_vs_train.csv", index=False)

    if DISPLAY_QC_TABLES or verbose:
        display(Markdown("### QC technique RNA — résumé"))
        display(rna_metric_summary)
        display(Markdown("### Décalage relatif des médianes vs train — RNA"))
        display(rna_shift.head(30))
        display(Markdown("### QC technique protéine — résumé"))
        display(prot_metric_summary)
        display(Markdown("### Décalage relatif des médianes vs train — protéine"))
        display(prot_shift.head(30))
    else:
        n_rna_flags = count_shift_flags(rna_shift)
        n_prot_flags = count_shift_flags(prot_shift)
        log_progress(f"QC technique : {n_rna_flags} flags RNA, {n_prot_flags} flags protéine (> {MAX_ACCEPTABLE_TECH_SHIFT_PCT:.0f}% vs train).")
        flagged = []
        if n_rna_flags:
            tmp = rna_shift[rna_shift["flag_abs_shift_gt_30pct"]].copy()
            tmp.insert(0, "modality", "rna")
            flagged.append(tmp)
        if n_prot_flags:
            tmp = prot_shift[prot_shift["flag_abs_shift_gt_30pct"]].copy()
            tmp.insert(0, "modality", "protein")
            flagged.append(tmp)
        if flagged:
            display(pd.concat(flagged, ignore_index=True).head(20))

    if MAKE_TECHNICAL_BOXPLOTS:
        plot_metric_boxplots(
            rna_metrics,
            rna.obs["split"],
            title=f"{dataset} — RNA technical metrics by split",
            path=fig_dir / f"{dataset}_03_rna_technical_boxplots_seed{seed}.png",
            max_metrics=6,
            show=make_plots,
        )
        plot_metric_boxplots(
            prot_metrics,
            prot.obs["split"],
            title=f"{dataset} — protein technical metrics by split",
            path=fig_dir / f"{dataset}_04_protein_technical_boxplots_seed{seed}.png",
            max_metrics=6,
            show=make_plots,
        )

    spatial_split_params = make_h5ad_safe({
        "dataset": dataset,
        "seed": seed,
        "unit_mode": cfg["unit_mode"],
        "unit_col": unit_col,
        "cell_buffer_width_um": CELL_BUFFER_WIDTH_UM,
        "connectivity_radius_factor": CONNECTIVITY_RADIUS_FACTOR,
        "split_info": split_info,
        "buffer_info": buffer_info,
    })

    rna.uns["spatial_split_params"] = spatial_split_params
    prot.uns["spatial_split_params"] = spatial_split_params.copy()

    if save_h5ad:
        rna.write_h5ad(h5ad_dir / f"{dataset}_rna_with_spatial_split_seed{seed}.h5ad")
        prot.write_h5ad(h5ad_dir / f"{dataset}_protein_with_spatial_split_seed{seed}.h5ad")

        keep = rna.obs["split"].astype(str) != "buffer"
        rna[keep].copy().write_h5ad(h5ad_dir / f"{dataset}_rna_train_val_test_no_buffer_seed{seed}.h5ad")
        prot[keep].copy().write_h5ad(h5ad_dir / f"{dataset}_protein_train_val_test_no_buffer_seed{seed}.h5ad")

    return {
        "dataset": dataset,
        "seed": seed,
        "rna": rna,
        "prot": prot,
        "unit_pos": unit_pos,
        "unit_split": unit_split,
        "split_summary": split_summary,
        "unit_summary": unit_summary,
        "rna_shift": rna_shift,
        "protein_shift": prot_shift,
        "split_info": split_info,
        "buffer_info": buffer_info,
    }


## 6. Annotation / clustering QC global avant le screening

Cette étape calcule une colonne biologique **une seule fois par dataset**, avant de tester les seeds.

- Si une colonne d'annotation existe déjà dans `obs`, elle est réutilisée.
- Sinon, le notebook lance un clustering Leiden CPU sur le RNA.
- La colonne finale est stockée dans `obs[CELLTYPE_COL]`.
- Pendant le screening, chaque seed est évaluée aussi avec la TVD de cette colonne entre train / val / test.

Cela évite de relancer Leiden pour chaque seed.


In [ ]:

# ============================================================
# Annotation / clustering QC global pour le screening de seeds
# ============================================================

# Marqueurs canoniques par type cellulaire.
# Ils servent uniquement à obtenir des labels QC approximatifs.
# Si les marqueurs ne sont pas suffisants, les clusters Leiden restent déjà utiles
# pour comparer les compositions train/val/test.

MARKERS_BREAST = {
    "Epithelial/Tumor":       ["EPCAM", "KRT8", "KRT18", "KRT19"],
    "Luminal epithelial":     ["ESR1", "GATA3", "FOXA1", "KRT8"],
    "Basal/Myoepithelial":    ["KRT5", "KRT14", "TP63", "ACTA2", "MYLK"],
    "T cells":                ["CD3D", "CD3E", "CD2", "TRAC"],
    "CD8 T cells":            ["CD8A", "CD8B"],
    "CD4 T cells":            ["CD4", "IL7R"],
    "B cells":                ["MS4A1", "CD79A", "CD79B"],
    "Plasma cells":           ["MZB1", "IGHG1", "JCHAIN", "SDC1"],
    "NK cells":               ["NKG7", "GNLY", "KLRD1"],
    "Macrophages/Myeloid":    ["CD68", "CD14", "LYZ", "CD163", "ITGAM"],
    "Dendritic cells":        ["CLEC9A", "ITGAX", "LILRA4"],
    "Mast cells":             ["TPSAB1", "CPA3", "KIT"],
    "Endothelial":            ["PECAM1", "VWF", "CDH5"],
    "Fibroblasts/CAFs":       ["COL1A1", "DCN", "LUM", "FAP", "PDGFRA"],
    "Pericytes":              ["RGS5", "PDGFRB", "ACTA2"],
    "Adipocytes":             ["ADIPOQ", "LEP"],
}

MARKERS_KIDNEY = {
    "Podocytes":                   ["NPHS1", "NPHS2", "PODXL", "PTPRO"],
    "Proximal tubule":             ["LRP2", "CUBN", "SLC34A1", "GATM", "MIOX"],
    "Loop of Henle (TAL)":         ["UMOD", "SLC12A1", "CASR"],
    "Distal convoluted tubule":    ["SLC12A3", "PVALB"],
    "Collecting duct principal":   ["AQP2", "AQP3"],
    "Collecting duct intercalated":["SLC26A4", "ATP6V0D2", "FOXI1"],
    "Endothelial":                 ["PECAM1", "VWF", "FLT1", "EMCN"],
    "Mesangial/JG cells":          ["PDGFRB", "ITGA8", "REN"],
    "Fibroblasts":                 ["COL1A1", "DCN", "PDGFRA"],
    "T cells":                     ["CD3D", "CD3E", "TRAC"],
    "B cells":                     ["MS4A1", "CD79A"],
    "Macrophages/Myeloid":         ["CD68", "LYZ", "CD14", "ITGAM"],
    "NK cells":                    ["NKG7", "GNLY"],
}

MARKERS_BY_DATASET = {
    "cosmx_breast": MARKERS_BREAST,
    "xenium_renal": MARKERS_KIDNEY,
}


def find_existing_celltype_col(adata: ad.AnnData, candidates=EXISTING_CELLTYPE_CANDIDATES) -> str | None:
    """Retourne la première colonne d'annotation plausible trouvée dans adata.obs."""
    for col in candidates:
        if col in adata.obs.columns:
            values = adata.obs[col].astype(str)
            if values.notna().any() and values.nunique(dropna=True) > 1:
                return col
    return None


def annotate_cpu_leiden_for_qc(
    adata: ad.AnnData,
    markers: dict[str, list[str]] | None = None,
    resolution: float = LEIDEN_RESOLUTION,
    n_hvg: int = CPU_ANNOTATION_N_HVG,
    n_pcs: int = CPU_ANNOTATION_N_PCS,
    n_neighbors: int = CPU_ANNOTATION_N_NEIGHBORS,
    celltype_col: str = CELLTYPE_COL,
    leiden_col: str = QC_LEIDEN_COL,
    force: bool = FORCE_RECOMPUTE_CELLTYPE_QC,
) -> ad.AnnData:
    """Clustering Leiden CPU une seule fois sur tout le dataset.

    Cette annotation est uniquement un outil QC pour comparer la composition
    biologique des splits. Elle ne doit pas être interprétée comme une annotation
    finale experte.

    Pour limiter la mémoire, on utilise peu de HVG et on évite l'affichage détaillé.
    """
    if (not force) and (celltype_col in adata.obs.columns):
        log_progress(f"  Colonne `{celltype_col}` déjà présente : annotation QC non recalculée.")
        return adata

    import scanpy as sc
    import numpy as np

    log_progress(f"  Annotation CPU QC : {adata.n_obs:,} cellules × {adata.n_vars:,} variables")
    log_progress("  Copie AnnData pour annotation QC...")
    a = adata.copy()

    log_progress("  Normalisation RNA : normalize_total...")
    sc.pp.normalize_total(a, target_sum=1e4)
    log_progress("  Transformation RNA : log1p...")
    sc.pp.log1p(a)
    norm = a.copy()  # log-normalisé, utilisé pour scorer les marqueurs

    n_top = min(int(n_hvg), a.n_vars)
    log_progress(f"  Sélection HVG : top {n_top} gènes variables...")
    sc.pp.highly_variable_genes(a, n_top_genes=n_top)
    if "highly_variable" in a.var.columns and a.var["highly_variable"].any():
        a = a[:, a.var["highly_variable"]].copy()
        log_progress(f"  HVG conservés : {a.n_vars:,} gènes.")

    # zero_center=False évite de densifier les matrices sparse dans la plupart des cas.
    # Pour un clustering QC, c'est suffisant et beaucoup plus léger.
    log_progress("  Scaling sparse-friendly : sc.pp.scale(zero_center=False)...")
    sc.pp.scale(a, max_value=10, zero_center=False)

    n_comp = min(int(n_pcs), a.n_vars - 1, a.n_obs - 1)
    if n_comp < 2:
        raise ValueError("Pas assez de variables/cellules pour calculer une PCA QC.")

    log_progress(f"  PCA QC : {n_comp} composantes...")
    sc.tl.pca(a, n_comps=n_comp, svd_solver="arpack")
    log_progress(f"  Graphe de voisins QC : n_neighbors={int(n_neighbors)}, n_pcs={n_comp}...")
    sc.pp.neighbors(a, n_neighbors=int(n_neighbors), n_pcs=n_comp)
    log_progress(f"  Leiden QC : resolution={float(resolution)}...")
    sc.tl.leiden(a, resolution=float(resolution), key_added=leiden_col)
    log_progress(f"  Leiden terminé : {a.obs[leiden_col].nunique()} clusters.")

    clusters = a.obs[leiden_col].astype(str)
    adata.obs[leiden_col] = clusters.reindex(adata.obs_names).astype(str).values

    if markers:
        cluster_label = {}
        for cl in sorted(clusters.unique(), key=str):
            mask = (clusters == cl).to_numpy()
            best_label = f"cluster_{cl}"
            best_score = -np.inf

            for ctype, genes in markers.items():
                present = [g for g in genes if g in norm.var_names]
                if len(present) == 0:
                    continue

                X = norm[mask, present].X
                score = float(X.mean())
                if score > best_score:
                    best_label = ctype
                    best_score = score

            cluster_label[cl] = best_label

        adata.obs[celltype_col] = adata.obs[leiden_col].astype(str).map(cluster_label).astype(str)
    else:
        adata.obs[celltype_col] = "cluster_" + adata.obs[leiden_col].astype(str)

    log_progress(f"  Annotation QC stockée dans obs['{celltype_col}'] ({adata.obs[celltype_col].nunique()} labels).")
    return adata


def ensure_global_celltype_qc(
    adata: ad.AnnData,
    dataset: str,
    markers: dict[str, list[str]] | None = None,
    celltype_col: str = CELLTYPE_COL,
) -> ad.AnnData:
    """Crée ou réutilise une colonne de composition biologique pour le screening."""
    existing = find_existing_celltype_col(adata)

    if (not FORCE_RECOMPUTE_CELLTYPE_QC) and existing is not None:
        adata.obs[celltype_col] = adata.obs[existing].astype(str).values
        log_progress(f"  Annotation existante réutilisée : `{existing}` -> `{celltype_col}`.")
        return adata

    if not RUN_GLOBAL_CPU_ANNOTATION_FOR_SEED_SCREENING:
        log_progress("  RUN_GLOBAL_CPU_ANNOTATION_FOR_SEED_SCREENING=False : pas de TVD biologique.")
        return adata

    return annotate_cpu_leiden_for_qc(
        adata,
        markers=markers,
        celltype_col=celltype_col,
        force=FORCE_RECOMPUTE_CELLTYPE_QC,
    )


def celltype_proportions_by_split(
    adata: ad.AnnData,
    celltype_col: str = CELLTYPE_COL,
    split_col: str = "split",
    splits: tuple[str, ...] = ("train", "val", "test"),
) -> pd.DataFrame:
    """Table des proportions de types/clusters par split."""
    if celltype_col not in adata.obs.columns:
        return pd.DataFrame()

    df = adata.obs[[split_col, celltype_col]].copy()
    df[split_col] = df[split_col].astype(str)
    df[celltype_col] = df[celltype_col].astype(str)
    df = df[df[split_col].isin(splits)]

    if df.empty:
        return pd.DataFrame()

    return pd.crosstab(df[celltype_col], df[split_col], normalize="columns")


def proportion_divergence_vs_train(prop_table: pd.DataFrame, ref: str = "train") -> dict[str, float]:
    """TVD de chaque split vs train. Proche de 0 = composition proche."""
    out = {}
    if prop_table.empty or ref not in prop_table.columns:
        return out

    for col in prop_table.columns:
        if col == ref:
            continue
        out[col] = float(0.5 * (prop_table[col] - prop_table[ref]).abs().sum())
    return out


def celltype_tvd_for_current_split(
    adata: ad.AnnData,
    celltype_col: str = CELLTYPE_COL,
    split_col: str = "split",
) -> dict[str, float]:
    """Calcule TVD val/test vs train pour la colonne biologique déjà présente."""
    prop = celltype_proportions_by_split(
        adata,
        celltype_col=celltype_col,
        split_col=split_col,
        splits=("train", "val", "test"),
    )
    return proportion_divergence_vs_train(prop, ref="train")


## 6. Screening des seeds et exécution finale


### 6.1 Screening de plusieurs seeds

Cette étape teste plusieurs seeds de split spatial.

Important méthodologiquement : la composition cellulaire n'est pas utilisée pour choisir la seed la plus similaire au train. La TVD biologique sert seulement de **filtre d'acceptabilité** : on rejette les seeds trop déséquilibrées si possible, puis on choisit parmi les seeds acceptables selon les critères principaux du split : tailles, buffer, QC technique et connexité.

Cela évite de fabriquer artificiellement un test trop proche du train tout en empêchant les splits biologiquement absurdes.

In [ ]:

def prepare_dataset_for_seed_screening(dataset: str, cfg: dict) -> dict:
    """Charge une fois le dataset, prépare les unités spatiales et calcule l'annotation QC globale."""
    log_progress(f"Préparation screening : {dataset}")
    log_progress(f"[{dataset}] Lecture RNA : {cfg['rna_path']}")
    rna = ad.read_h5ad(cfg["rna_path"])
    log_progress(f"[{dataset}] Lecture protéine : {cfg['protein_path']}")
    prot = ad.read_h5ad(cfg["protein_path"])
    log_progress(f"[{dataset}] Données chargées : RNA {rna.n_obs:,} × {rna.n_vars:,}, Protein {prot.n_obs:,} × {prot.n_vars:,}")
    log_progress(f"[{dataset}] Harmonisation RNA/protéine...")
    rna, prot = harmonize_pair(rna, prot)
    log_progress(f"[{dataset}] Préparation coordonnées spatiales en µm...")
    rna, prot = prepare_spatial_um(rna, prot, cfg, dataset)

    unit_col = cfg["unit_col"]
    if cfg["unit_mode"] == "pseudo_fov":
        rna, prot = add_pseudo_fov_from_grid(
            rna,
            prot,
            unit_col=unit_col,
            tile_size_um=float(cfg["pseudo_fov_tile_size_um"]),
        )
    elif cfg["unit_mode"] == "native_fov":
        if unit_col not in rna.obs.columns:
            raise ValueError(f"{dataset}: colonne FOV native absente : {unit_col}")
        for a in [rna, prot]:
            a.obs[unit_col] = a.obs[unit_col].astype(str)
        log_progress(f"[{dataset}] FOV natifs trouvés : {rna.obs[unit_col].nunique()} unités")
    else:
        raise ValueError(f"unit_mode inconnu : {cfg['unit_mode']}")

    log_progress(f"[{dataset}] Construction des positions d’unités spatiales...")
    unit_pos = build_unit_positions(rna, unit_col=unit_col)
    log_progress(f"[{dataset}] Positions d’unités prêtes : {len(unit_pos):,} unités.")

    # Annotation / clustering QC global, une seule fois avant le screening de seeds.
    markers = MARKERS_BY_DATASET.get(dataset, None)
    log_progress(f"[{dataset}] Début annotation/clustering QC global pour TVD biologique...")
    rna = ensure_global_celltype_qc(rna, dataset=dataset, markers=markers, celltype_col=CELLTYPE_COL)
    log_progress(f"[{dataset}] Annotation/clustering QC global terminé.")

    # Même colonne côté protéine pour garder l'appariement et sauvegarder des h5ad cohérents.
    if CELLTYPE_COL in rna.obs.columns:
        prot.obs[CELLTYPE_COL] = rna.obs[CELLTYPE_COL].astype(str).values
    if QC_LEIDEN_COL in rna.obs.columns:
        prot.obs[QC_LEIDEN_COL] = rna.obs[QC_LEIDEN_COL].astype(str).values

    # Les métriques techniques et l'annotation ne dépendent pas de la seed : on les calcule une seule fois.
    log_progress(f"[{dataset}] Calcul métriques techniques RNA...")
    rna_metrics = get_obs_technical_metrics(rna, modality="rna")
    log_progress(f"[{dataset}] Calcul métriques techniques protéine...")
    prot_metrics = get_obs_technical_metrics(prot, modality="protein")

    if SAVE_GLOBAL_CELLTYPE_QC and CELLTYPE_COL in rna.obs.columns:
        out_dir = OUTPUT_DIR / dataset / "tables"
        out_dir.mkdir(parents=True, exist_ok=True)
        global_counts = (
            rna.obs[CELLTYPE_COL]
            .astype(str)
            .value_counts(normalize=False)
            .rename_axis(CELLTYPE_COL)
            .reset_index(name="n_cells")
        )
        global_counts["pct_cells"] = 100 * global_counts["n_cells"] / rna.n_obs
        global_counts.to_csv(out_dir / f"{dataset}_global_{CELLTYPE_COL}_composition.csv", index=False)
        log_progress(f"[{dataset}] Composition globale sauvegardée : {out_dir / f'{dataset}_global_{CELLTYPE_COL}_composition.csv'}")

    log_progress(f"[{dataset}] Préparation screening terminée.")

    return {
        "dataset": dataset,
        "cfg": cfg,
        "rna": rna,
        "prot": prot,
        "unit_col": unit_col,
        "unit_pos": unit_pos,
        "rna_metrics": rna_metrics,
        "prot_metrics": prot_metrics,
    }


def celltype_tvd_status(max_tvd: float) -> str:
    """Statut QC biologique utilisé comme filtre, pas comme score continu."""
    if not np.isfinite(max_tvd):
        return "MISSING"
    if max_tvd <= CELLTYPE_TVD_WARNING:
        return "OK"
    if max_tvd <= MAX_ACCEPTABLE_CELLTYPE_TVD:
        return "WARNING"
    return "REJECT"


def score_seed_qc(row: dict) -> float:
    """Score heuristique principal, SANS optimiser la TVD biologique.

    Plus petit = meilleur compromis spatial/technique.

    La TVD cellulaire est traitée ailleurs comme filtre d'acceptabilité :
    elle sert à rejeter les splits biologiquement absurdes, mais pas à choisir
    la seed qui ressemble le plus au train.
    """
    val_target_pct = 100 * SPLIT_FRACTIONS["val"]
    test_target_pct = 100 * SPLIT_FRACTIONS["test"]

    val_dev = abs(row["val_pct"] - val_target_pct)
    test_dev = abs(row["test_pct"] - test_target_pct)

    buffer_penalty = max(0.0, row["buffer_pct"] - MAX_ACCEPTABLE_BUFFER_PCT)
    small_val_test_penalty = (
        max(0.0, MIN_ACCEPTABLE_VAL_TEST_PCT - row["val_pct"])
        + max(0.0, MIN_ACCEPTABLE_VAL_TEST_PCT - row["test_pct"])
    )

    tech_penalty = 2.0 * (row["rna_n_flags"] + row["protein_n_flags"])
    shift_penalty = 0.0
    if np.isfinite(row["max_abs_shift_pct"]):
        shift_penalty = max(0.0, row["max_abs_shift_pct"] - MAX_ACCEPTABLE_TECH_SHIFT_PCT) / 5.0

    cc_penalty = row["cc_reassigned_pct"]

    return float(
        val_dev
        + test_dev
        + 0.30 * row["buffer_pct"]
        + 3.0 * buffer_penalty
        + 5.0 * small_val_test_penalty
        + tech_penalty
        + shift_penalty
        + 0.50 * cc_penalty
    )


def evaluate_seed_on_prepared_dataset(prepared: dict, seed: int) -> dict:
    dataset = prepared["dataset"]
    cfg = prepared["cfg"]
    rna = prepared["rna"]
    prot = prepared["prot"]
    unit_col = prepared["unit_col"]
    unit_pos = prepared["unit_pos"]

    unit_split, cc_report, split_info = make_unit_split(unit_pos, cfg, seed=seed)
    rna, prot = apply_base_split_to_cells(rna, prot, unit_col=unit_col, unit_split=unit_split)
    rna, prot, buffer_info = add_cell_level_boundary_buffer(
        rna,
        prot,
        width_um=CELL_BUFFER_WIDTH_UM,
        base_split_col="unit_base_split",
        output_col="split",
    )

    split_summary = split_counts(rna, split_col="split")
    pcts = split_summary["pct_cells"].to_dict()
    n_cells = split_summary["n_cells"].to_dict()

    rna_metric_summary = summarize_metrics_by_split(prepared["rna_metrics"], rna.obs["split"])
    prot_metric_summary = summarize_metrics_by_split(prepared["prot_metrics"], prot.obs["split"])
    rna_shift = median_shift_report(rna_metric_summary, reference_split="train")
    prot_shift = median_shift_report(prot_metric_summary, reference_split="train")

    rna_n_flags = count_shift_flags(rna_shift)
    prot_n_flags = count_shift_flags(prot_shift)
    max_shift = np.nanmax([max_abs_shift_pct(rna_shift), max_abs_shift_pct(prot_shift)])

    tvd = celltype_tvd_for_current_split(rna, celltype_col=CELLTYPE_COL, split_col="split")
    tvd_val = float(tvd.get("val", np.nan))
    tvd_test = float(tvd.get("test", np.nan))
    max_celltype_tvd = np.nanmax([tvd_val, tvd_test])
    celltype_status = celltype_tvd_status(max_celltype_tvd)
    celltype_acceptable = celltype_status in {"OK", "WARNING"}

    if cc_report.empty:
        cc_reassigned_cells = 0
    else:
        cc_reassigned_cells = int(
            cc_report.loc[
                cc_report["action"].astype(str).str.contains("reassign"),
                "n_cells",
            ].sum()
        )

    row = {
        "dataset": dataset,
        "seed": seed,
        "val_target_norm": tuple(round(x, 3) for x in split_info["val_target_norm"]),
        "test_target_norm": tuple(round(x, 3) for x in split_info["test_target_norm"]),
        "train_pct": float(pcts.get("train", 0.0)),
        "val_pct": float(pcts.get("val", 0.0)),
        "test_pct": float(pcts.get("test", 0.0)),
        "buffer_pct": float(pcts.get("buffer", 0.0)),
        "train_n": int(n_cells.get("train", 0)),
        "val_n": int(n_cells.get("val", 0)),
        "test_n": int(n_cells.get("test", 0)),
        "buffer_n": int(n_cells.get("buffer", 0)),
        "rna_n_flags": rna_n_flags,
        "protein_n_flags": prot_n_flags,
        "max_abs_shift_pct": float(max_shift) if np.isfinite(max_shift) else np.nan,
        "celltype_tvd_val": tvd_val,
        "celltype_tvd_test": tvd_test,
        "max_celltype_tvd": float(max_celltype_tvd) if np.isfinite(max_celltype_tvd) else np.nan,
        "celltype_tvd_status": celltype_status,
        "celltype_tvd_warning": bool(np.isfinite(max_celltype_tvd) and max_celltype_tvd > CELLTYPE_TVD_WARNING),
        "celltype_tvd_acceptable": bool(celltype_acceptable),
        "cc_reassigned_cells": cc_reassigned_cells,
        "cc_reassigned_pct": 100 * cc_reassigned_cells / rna.n_obs,
    }
    row["qc_score_without_celltype"] = score_seed_qc(row)
    # Alias conservé pour compatibilité avec le reste du notebook.
    # Il n'inclut PAS de pénalité continue liée à la TVD biologique.
    row["qc_score"] = row["qc_score_without_celltype"]
    return row


def screen_spatial_split_seeds(
    datasets: dict,
    seeds: list[int] | tuple[int, ...],
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """Teste plusieurs seeds et renvoie le détail par dataset + un résumé global + objets préparés."""
    out_dir = OUTPUT_DIR / "seed_screening"
    out_dir.mkdir(parents=True, exist_ok=True)

    detailed_rows = []
    prepared_by_dataset = {}

    for dataset, cfg in datasets.items():
        prepared_by_dataset[dataset] = prepare_dataset_for_seed_screening(dataset, cfg)

        for seed in seeds:
            row = evaluate_seed_on_prepared_dataset(prepared_by_dataset[dataset], int(seed))
            detailed_rows.append(row)

    detailed = pd.DataFrame(detailed_rows)
    detailed = detailed.sort_values(["seed", "dataset"]).reset_index(drop=True)
    detailed.to_csv(out_dir / "seed_screening_by_dataset.csv", index=False)
    log_progress(f"Table détaillée seed screening sauvegardée : {out_dir / 'seed_screening_by_dataset.csv'}")

    summary = (
        detailed.groupby("seed")
        .agg(
            qc_score_mean=("qc_score_without_celltype", "mean"),
            qc_score_max=("qc_score_without_celltype", "max"),
            max_buffer_pct=("buffer_pct", "max"),
            min_val_pct=("val_pct", "min"),
            min_test_pct=("test_pct", "min"),
            total_technical_flags=("rna_n_flags", "sum"),
            total_protein_flags=("protein_n_flags", "sum"),
            max_abs_shift_pct=("max_abs_shift_pct", "max"),
            max_celltype_tvd=("max_celltype_tvd", "max"),
            any_celltype_tvd_warning=("celltype_tvd_warning", "max"),
            all_celltype_tvd_acceptable=("celltype_tvd_acceptable", "min"),
            max_cc_reassigned_pct=("cc_reassigned_pct", "max"),
        )
        .reset_index()
    )
    summary["total_flags"] = summary["total_technical_flags"] + summary["total_protein_flags"]

    # Filtres d'acceptabilité.
    # La TVD cellulaire n'est PAS utilisée pour trier les seeds acceptables :
    # elle sert seulement à exclure les splits trop déséquilibrés biologiquement.
    summary["celltype_filter_pass"] = summary["max_celltype_tvd"] <= MAX_ACCEPTABLE_CELLTYPE_TVD
    summary["size_filter_pass"] = (
        (summary["min_val_pct"] >= MIN_ACCEPTABLE_VAL_TEST_PCT)
        & (summary["min_test_pct"] >= MIN_ACCEPTABLE_VAL_TEST_PCT)
    )
    summary["buffer_filter_pass"] = summary["max_buffer_pct"] <= MAX_ACCEPTABLE_BUFFER_PCT
    summary["hard_filters_pass"] = (
        summary["celltype_filter_pass"]
        & summary["size_filter_pass"]
        & summary["buffer_filter_pass"]
    )

    summary = summary.sort_values(
        ["hard_filters_pass", "qc_score_mean", "qc_score_max", "total_flags", "max_abs_shift_pct"],
        ascending=[False, True, True, True, True],
    ).reset_index(drop=True)
    summary.to_csv(out_dir / "seed_screening_summary.csv", index=False)
    log_progress(f"Résumé seed screening sauvegardé : {out_dir / 'seed_screening_summary.csv'}")

    return detailed, summary, prepared_by_dataset


def select_best_seed_from_screening_summary(summary: pd.DataFrame) -> tuple[int, str]:
    """Sélectionne la seed finale sans optimiser la TVD biologique.

    Priorité :
    1. utiliser uniquement les seeds qui passent les filtres d'acceptabilité,
       dont le filtre TVD ;
    2. parmi elles, choisir selon le score spatial/technique sans TVD ;
    3. si aucune seed ne passe tous les filtres, fallback explicite.
    """
    if summary.empty:
        raise ValueError("screening_summary est vide : impossible de choisir une seed.")

    pool = summary[summary["hard_filters_pass"]].copy()
    reason = "seed choisie parmi celles qui passent les filtres taille/buffer/TVD, puis triée par QC spatial/technique"

    if pool.empty:
        # Fallback 1 : ne pas optimiser la TVD, mais au moins éviter les seeds biologiquement rejetées.
        pool = summary[summary["celltype_filter_pass"]].copy()
        reason = (
            "aucune seed ne passe tous les filtres ; seed choisie parmi celles qui passent le filtre TVD, "
            "puis triée par QC spatial/technique"
        )

    if pool.empty:
        # Fallback 2 : toutes les seeds échouent au filtre TVD ; on garde la meilleure techniquement,
        # mais il faudra inspecter le split manuellement.
        pool = summary.copy()
        reason = (
            "aucune seed ne passe le filtre TVD ; seed choisie par QC spatial/technique uniquement, "
            "à inspecter manuellement"
        )

    pool = pool.sort_values(
        ["qc_score_mean", "qc_score_max", "total_flags", "max_abs_shift_pct"],
        ascending=[True, True, True, True],
    )
    return int(pool.iloc[0]["seed"]), reason


def process_final_split_from_prepared(
    prepared: dict,
    seed: int,
    display_mode: str = DISPLAY_MODE,
    make_plots: bool = SHOW_PLOTS,
    save_h5ad: bool = SAVE_H5AD,
):
    """Exécute le split final depuis les objets déjà préparés/annotés."""
    dataset = prepared["dataset"]
    cfg = prepared["cfg"]
    rna = prepared["rna"]
    prot = prepared["prot"]
    unit_col = prepared["unit_col"]
    unit_pos = prepared["unit_pos"]

    verbose = display_mode == "full"

    dataset_out = OUTPUT_DIR / dataset
    fig_dir = dataset_out / "figures"
    table_dir = dataset_out / "tables"
    h5ad_dir = dataset_out / "h5ad"
    for d in [fig_dir, table_dir, h5ad_dir]:
        d.mkdir(parents=True, exist_ok=True)

    display(Markdown(f"## Dataset final : `{dataset}` — seed `{seed}`"))
    log_progress(f"[{dataset}] Split final : RNA {rna.n_obs:,} cellules × {rna.n_vars:,} variables")
    log_progress(f"[{dataset}] Split final : Protein {prot.n_obs:,} cellules × {prot.n_vars:,} variables")

    unit_pos.to_csv(table_dir / f"{dataset}_unit_positions.csv", index=False)

    unit_split, cc_report, split_info = make_unit_split(unit_pos, cfg, seed=seed)
    cc_report.to_csv(table_dir / f"{dataset}_connected_components_report_seed{seed}.csv", index=False)

    log_progress(f"[{dataset}] Cible val : {tuple(round(x, 3) for x in split_info['val_target_norm'])}")
    log_progress(f"[{dataset}] Cible test : {tuple(round(x, 3) for x in split_info['test_target_norm'])}")

    plot_unit_split(
        unit_pos,
        unit_split,
        title=f"{dataset} — split des unités spatiales — seed {seed}",
        path=fig_dir / f"{dataset}_01_unit_split_seed{seed}.png",
        show=make_plots,
    )

    rna, prot = apply_base_split_to_cells(rna, prot, unit_col=unit_col, unit_split=unit_split)
    rna, prot, buffer_info = add_cell_level_boundary_buffer(
        rna,
        prot,
        width_um=CELL_BUFFER_WIDTH_UM,
        base_split_col="unit_base_split",
        output_col="split",
    )

    split_summary = split_counts(rna, split_col="split")
    split_summary.to_csv(table_dir / f"{dataset}_split_cell_counts_seed{seed}.csv")

    display(Markdown("### Résumé final par cellule"))
    display(split_summary)

    unit_summary = unit_counts(rna, unit_col=unit_col, split_col="unit_base_split")
    unit_summary.to_csv(table_dir / f"{dataset}_unit_base_split_counts_seed{seed}.csv")

    if verbose:
        log_progress(f"[{dataset}] Informations buffer cellule : {buffer_info}")

    plot_spatial_split(
        rna,
        title=f"{dataset} — split final avec buffer cellule — seed {seed}",
        path=fig_dir / f"{dataset}_02_cell_split_final_seed{seed}.png",
        split_col="split",
        show=make_plots,
    )

    # QC distributions techniques RNA
    rna_metrics = prepared["rna_metrics"]
    rna_metric_summary = summarize_metrics_by_split(rna_metrics, rna.obs["split"])
    rna_metric_summary.to_csv(table_dir / f"{dataset}_rna_technical_metrics_by_split_seed{seed}.csv", index=False)

    rna_shift = median_shift_report(rna_metric_summary, reference_split="train")
    rna_shift.to_csv(table_dir / f"{dataset}_rna_median_shift_vs_train_seed{seed}.csv", index=False)

    # QC distributions techniques protéine
    prot_metrics = prepared["prot_metrics"]
    prot_metric_summary = summarize_metrics_by_split(prot_metrics, prot.obs["split"])
    prot_metric_summary.to_csv(table_dir / f"{dataset}_protein_technical_metrics_by_split_seed{seed}.csv", index=False)

    prot_shift = median_shift_report(prot_metric_summary, reference_split="train")
    prot_shift.to_csv(table_dir / f"{dataset}_protein_median_shift_vs_train_seed{seed}.csv", index=False)

    n_rna_flags = count_shift_flags(rna_shift)
    n_prot_flags = count_shift_flags(prot_shift)
    log_progress(f"[{dataset}] QC technique : {n_rna_flags} flags RNA, {n_prot_flags} flags protéine (> {MAX_ACCEPTABLE_TECH_SHIFT_PCT:.0f}% vs train).")

    # QC biologique final, sans recalculer l'annotation
    prop = celltype_proportions_by_split(rna, celltype_col=CELLTYPE_COL, split_col="split")
    tvd = proportion_divergence_vs_train(prop)

    if not prop.empty:
        prop_pct = (100 * prop).round(3)
        prop_pct.to_csv(table_dir / f"{dataset}_celltype_proportions_by_split_seed{seed}.csv")
        rna.obs[[CELLTYPE_COL, "split"]].to_csv(table_dir / f"{dataset}_celltype_annotations_seed{seed}.csv")

        tvd_rows = []
        for split_name, value in tvd.items():
            flag = "OK" if value <= MAX_ACCEPTABLE_CELLTYPE_TVD else "A VERIFIER"
            log_progress(f"[{dataset}] TVD {split_name} vs train = {value:.3f} -> {flag}")
            tvd_rows.append({
                "dataset": dataset,
                "seed": seed,
                "split": split_name,
                "tvd_vs_train": value,
                "flag": flag,
            })
        pd.DataFrame(tvd_rows).to_csv(table_dir / f"{dataset}_celltype_TVD_seed{seed}.csv", index=False)

        if DISPLAY_CELLTYPE_TABLES:
            display(Markdown(f"### Proportions de types par split — {dataset}"))
            display(prop_pct.round(2))
    else:
        log_progress(f"[{dataset}] Aucune colonne `{CELLTYPE_COL}` disponible : TVD biologique non calculée.")

    spatial_split_params = make_h5ad_safe({
        "dataset": dataset,
        "seed": seed,
        "unit_mode": cfg["unit_mode"],
        "unit_col": unit_col,
        "cell_buffer_width_um": CELL_BUFFER_WIDTH_UM,
        "connectivity_radius_factor": CONNECTIVITY_RADIUS_FACTOR,
        "celltype_col_for_qc": CELLTYPE_COL if CELLTYPE_COL in rna.obs.columns else None,
        "celltype_tvd_vs_train": tvd,
        "split_info": split_info,
        "buffer_info": buffer_info,
    })

    rna.uns["spatial_split_params"] = spatial_split_params
    prot.uns["spatial_split_params"] = spatial_split_params.copy()

    if save_h5ad:
        log_progress(f"[{dataset}] Sauvegarde h5ad avec buffer...")
        rna.write_h5ad(h5ad_dir / f"{dataset}_rna_with_spatial_split_seed{seed}.h5ad")
        prot.write_h5ad(h5ad_dir / f"{dataset}_protein_with_spatial_split_seed{seed}.h5ad")

        keep = rna.obs["split"].astype(str) != "buffer"
        log_progress(f"[{dataset}] Sauvegarde h5ad sans buffer...")
        rna[keep].copy().write_h5ad(h5ad_dir / f"{dataset}_rna_train_val_test_no_buffer_seed{seed}.h5ad")
        prot[keep].copy().write_h5ad(h5ad_dir / f"{dataset}_protein_train_val_test_no_buffer_seed{seed}.h5ad")
        log_progress(f"[{dataset}] Sauvegarde h5ad terminée.")

    return {
        "dataset": dataset,
        "seed": seed,
        "rna": rna,
        "prot": prot,
        "unit_pos": unit_pos,
        "unit_split": unit_split,
        "split_summary": split_summary,
        "unit_summary": unit_summary,
        "rna_shift": rna_shift,
        "protein_shift": prot_shift,
        "celltype_proportions": prop,
        "celltype_tvd": tvd,
        "split_info": split_info,
        "buffer_info": buffer_info,
    }


In [ ]:

if RUN_SEED_SCREENING:
    log_progress("RUN_SEED_SCREENING=True : lancement screening des seeds.")
    seed_screening_by_dataset, seed_screening_summary, prepared_by_dataset = screen_spatial_split_seeds(
        DATASETS,
        SEEDS_TO_TEST,
    )

    display(Markdown("### Résumé compact du screening de seeds"))
    cols_summary = [
        "seed",
        "hard_filters_pass",
        "celltype_filter_pass",
        "max_celltype_tvd",
        "qc_score_mean",
        "qc_score_max",
        "max_buffer_pct",
        "min_val_pct",
        "min_test_pct",
        "total_flags",
        "max_abs_shift_pct",
        "max_cc_reassigned_pct",
    ]
    display(seed_screening_summary[cols_summary].round(3))

    best_seed_candidate, selection_reason = select_best_seed_from_screening_summary(seed_screening_summary)
    log_progress(f"Seed candidate selon le QC global du split : {best_seed_candidate}")
    log_progress(f"Règle de sélection : {selection_reason}")
    log_progress("Important : la TVD biologique est un filtre d'acceptabilité, pas un score à minimiser.")
    log_progress(f"Détail sauvegardé dans : {(OUTPUT_DIR / 'seed_screening').resolve()}")

    if FINAL_SEED is None:
        SEED = best_seed_candidate
    else:
        SEED = int(FINAL_SEED)
        log_progress(f"FINAL_SEED défini manuellement : {SEED}")

    display(Markdown("### Détail de la seed finalement utilisée"))
    selected_seed_details = seed_screening_by_dataset[seed_screening_by_dataset["seed"] == SEED].copy()
    cols_detail = [
        "dataset",
        "seed",
        "qc_score_without_celltype",
        "train_pct",
        "val_pct",
        "test_pct",
        "buffer_pct",
        "celltype_tvd_val",
        "celltype_tvd_test",
        "max_celltype_tvd",
        "celltype_tvd_status",
        "celltype_tvd_acceptable",
        "rna_n_flags",
        "protein_n_flags",
        "max_abs_shift_pct",
        "cc_reassigned_pct",
    ]
    display(selected_seed_details[cols_detail].round(3))
else:
    seed_screening_by_dataset = pd.DataFrame()
    seed_screening_summary = pd.DataFrame()
    prepared_by_dataset = {}

    if FINAL_SEED is not None:
        SEED = int(FINAL_SEED)

log_progress(f"Seed utilisée pour le split final : {SEED}")

results = {}

if RUN_FINAL_SPLIT_AFTER_SCREENING:
    log_progress("RUN_FINAL_SPLIT_AFTER_SCREENING=True : lancement des splits finaux.")
    if prepared_by_dataset:
        for dataset in ["cosmx_breast", "xenium_renal"]:
            log_progress(f"Début split final : {dataset}")
            results[dataset] = process_final_split_from_prepared(
                prepared_by_dataset[dataset],
                seed=SEED,
                display_mode=DISPLAY_MODE,
                make_plots=SHOW_PLOTS,
                save_h5ad=SAVE_H5AD,
            )
    else:
        # Fallback si le screening est désactivé : ancien pipeline.
        for dataset in ["cosmx_breast", "xenium_renal"]:
            log_progress(f"Début split final fallback : {dataset}")
            results[dataset] = process_one_dataset(
                dataset,
                DATASETS[dataset],
                seed=SEED,
                display_mode=DISPLAY_MODE,
                make_plots=SHOW_PLOTS,
                save_h5ad=SAVE_H5AD,
            )

    log_progress("Splits finaux terminés.")
    log_progress(f"Dossier de sortie : {OUTPUT_DIR.resolve()}")
else:
    results = {}
    log_progress("RUN_FINAL_SPLIT_AFTER_SCREENING=False : aucun split final sauvegardé.")



Préparation screening : cosmx_breast
[cosmx_breast] Coordonnées : obsm['spatial']
[cosmx_breast] Facteur conversion vers µm : 0.120280945
[cosmx_breast] bbox µm : x=[544.8, 7195.6], y=[10256.8, 14861.3]
FOV natifs trouvés : 108 unités
  Annotation CPU QC : 152,451 cellules × 20,385 variables


/opt/homebrew/Caskroom/miniforge/base/envs/rna_protein_clip/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/vr/bmk77xkx3yj90ccpc3jv7f_c0000gp/T/ipykernel_23195/3636569431.py:109: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(a, resolution=float(resolution), key_added=leiden_col)


  Annotation QC stockée dans obs['qc_celltype_cpu'] (8 labels).

Préparation screening : xenium_renal
[xenium_renal] Coordonnées : obsm['spatial']
[xenium_renal] Facteur conversion vers µm : 1.0
[xenium_renal] bbox µm : x=[1.4, 11472.9], y=[3.3, 5805.0]
Pseudo-FOV créés : 111 tuiles de 800 µm
  Annotation CPU QC : 465,534 cellules × 405 variables


/opt/homebrew/Caskroom/miniforge/base/envs/rna_protein_clip/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


KeyboardInterrupt: 

## 7. Résumé global

In [ ]:

if results:
    log_progress("Création des résumés globaux finaux.")
    all_split_summaries = []
    all_shift_flags = []
    all_celltype_tvd = []

    for dataset, res in results.items():
        tmp = res["split_summary"].reset_index().rename(columns={"index": "split"})
        tmp.insert(0, "dataset", dataset)
        tmp.insert(1, "seed", res.get("seed", SEED))
        all_split_summaries.append(tmp)

        for modality, key in [("rna", "rna_shift"), ("protein", "protein_shift")]:
            shift = res[key].copy()
            if not shift.empty:
                shift.insert(0, "modality", modality)
                shift.insert(0, "dataset", dataset)
                all_shift_flags.append(shift)

        tvd = res.get("celltype_tvd", {})
        for split_name, value in tvd.items():
            all_celltype_tvd.append({
                "dataset": dataset,
                "seed": res.get("seed", SEED),
                "split": split_name,
                "tvd_vs_train": value,
                "status": celltype_tvd_status(value),
                "flag": "OK" if value <= CELLTYPE_TVD_WARNING else ("A VERIFIER" if value <= MAX_ACCEPTABLE_CELLTYPE_TVD else "REJET SI POSSIBLE"),
            })

    summary_all = pd.concat(all_split_summaries, ignore_index=True)
    display(Markdown("### Pourcentages finaux par dataset"))
    display(summary_all)
    summary_all.to_csv(OUTPUT_DIR / "global_split_cell_counts.csv", index=False)
    log_progress(f"Résumé global split sauvegardé : {OUTPUT_DIR / 'global_split_cell_counts.csv'}")

    if all_celltype_tvd:
        celltype_tvd_final = pd.DataFrame(all_celltype_tvd)
        display(Markdown("### TVD biologique finale"))
        display(celltype_tvd_final.round({"tvd_vs_train": 3}))
        celltype_tvd_final.to_csv(OUTPUT_DIR / "global_celltype_TVD_final.csv", index=False)
        log_progress(f"TVD finale sauvegardée : {OUTPUT_DIR / 'global_celltype_TVD_final.csv'}")

    if all_shift_flags:
        shift_all = pd.concat(all_shift_flags, ignore_index=True)
        flagged = shift_all[shift_all["flag_abs_shift_gt_30pct"]].copy()
        shift_all.to_csv(OUTPUT_DIR / "global_technical_median_shift_vs_train.csv", index=False)
        log_progress(f"Shifts techniques globaux sauvegardés : {OUTPUT_DIR / 'global_technical_median_shift_vs_train.csv'}")

        display(Markdown("### Résumé compact des flags techniques"))
        compact_flags = (
            flagged.groupby(["dataset", "modality", "split"])
            .size()
            .rename("n_flagged_metrics")
            .reset_index()
            if not flagged.empty
            else pd.DataFrame(columns=["dataset", "modality", "split", "n_flagged_metrics"])
        )
        display(compact_flags)

        if DISPLAY_QC_TABLES and not flagged.empty:
            display(Markdown("### Détail des métriques techniques avec décalage médian > 30% vs train"))
            display(flagged)
    else:
        log_progress("Aucun rapport de shift disponible.")
else:
    log_progress("Aucun résultat final disponible. Vérifie RUN_FINAL_SPLIT_AFTER_SCREENING.")


## Notes d'interprétation

- Le split final doit être choisi **avant** l'entraînement du modèle.
- La seed ne doit jamais être choisie en fonction des performances du modèle sur test.
- La TVD biologique est utilisée comme **filtre QC** : elle sert à éviter les splits où une région val/test a une composition cellulaire clairement aberrante.
- Elle ne doit pas être minimisée à tout prix, sinon le test risque d'être artificiellement trop proche du train.
- La seed finale est donc choisie ainsi :
  1. on garde les seeds qui passent les filtres taille / buffer / TVD ;
  2. parmi ces seeds acceptables, on choisit selon le score spatial/technique sans pénalité TVD ;
  3. si aucune seed ne passe tous les filtres, le notebook affiche un fallback à inspecter manuellement.
- Le train peut être non connexe : ce n'est pas forcément un problème si val/test sont des régions compactes et séparées par un buffer.
- `buffer` doit être exclu de l'entraînement et de l'évaluation finale.
- Les annotations `qc_celltype_cpu` sont des labels approximatifs de QC, pas des annotations biologiques définitives.